# Bronze Layer: Load Raw Data
Loads all CRM and ERP CSV files from the `source_systems` volume into bronze tables.
- **Source:** `/Volumes/workspace/bronze/source_systems/`
- **Target:** `workspace.bronze`
- **Load type:** Full load (overwrite)
- **Rule:** No cleaning in bronze. Data is kept exactly as in the source.

In [0]:
catalog     = "workspace"
schema      = "bronze"
volume_path = f"/Volumes/{catalog}/{schema}/source_systems"

sources = [
    # CRM
    {"file": "source_crm/cust_info.csv",     "table": "crm_cust_info"},
    {"file": "source_crm/prd_info.csv",      "table": "crm_prd_info"},
    {"file": "source_crm/sales_details.csv", "table": "crm_sales_details"},
    # ERP
    {"file": "source_erp/CUST_AZ12.csv",     "table": "erp_cust_az12"},
    {"file": "source_erp/LOC_A101.csv",      "table": "erp_loc_a101"},
    {"file": "source_erp/PX_CAT_G1V2.csv",   "table": "erp_px_cat_g1v2"},
]

## 2. Load files into bronze tables
Loops through the source list, reads each CSV, and saves it as a Delta table.
`overwrite` replaces the table on every run, so the notebook is safe to rerun.

In [0]:
for s in sources:
    path  = f"{volume_path}/{s['file']}"
    table = f"{catalog}.{schema}.{s['table']}"

    df = (spark.read
          .option("header", "true")
          .csv(path))

    df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(table)
    print(f"Loaded {table}: {df.count()} rows")

## 3. Check results
Lists all tables in the bronze schema to confirm the load.

In [0]:
%sql
SHOW TABLES IN workspace.bronze;